# Report figures and tables

Every figure and table in the report, generated from the saved backtests in `results/`
(produced by `python run_experiments.py all`). Figures are shown here and saved to `../report/figures/`.

* **S0**: direct index without loss harvesting (baseline)
* **S1**: TAM optimizer of Moehle et al. (2021) with gamma_risk = 400, chosen in-sample (2000-2012) so that tracking error is close to 0.5%
* **S2**: S1 with the deferral-aware tax term T = (rho_now − delta·rho_LT)(1 − basis/price), delta_t = 1.05^−(days to the end date / 365)

Main results, out-of-sample and the 6-year windows use gamma_risk = 400. S1 matched (gamma_risk = 800, chosen in-sample) has about the same tracking error as S2. The tracking-error frontier is a separate experiment: a cap on ex-ante tracking error with gamma_risk = 10, relaxed from 0.25% to 2%.

In [1]:
import os, glob, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tlh.metrics import compare, summary, pretax_returns, monthly, after_tax_liquidation_value

FIG = "../report/figures/"
os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True, "grid.alpha": 0.3})
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=["#1F3B5C", "#7FA6C9", "#9E9E9E"])   # navy, light steel blue, gray

def load(name):
    with open(f"results/{name}.pkl", "rb") as f:
        return pickle.load(f)

def save(fig, name):
    fig.savefig(FIG + name, dpi=200, bbox_inches="tight")

main = {k: load(f"main_{k}") for k in ["S0", "S1", "S2"]}
s0, s1 = main["S0"], main["S1"]

## Table 1. Main results, 2000–2025

In [2]:
T = compare(main)
rows = {"Pre-tax return": "pretax", "Index return": "index",
        "After-tax return, no liquidation": "after_tax_noliq", "After-tax return, liquidation": "after_tax_liq",
        "Tax alpha, no liquidation": "tax_alpha_noliq", "Tax alpha, liquidation": "tax_alpha_liq",
        "Tracking error, realized": "te_realized", "Tracking error, ex-ante (mean)": "te_ex_ante",
        "Sales turnover per year": "sell_turnover", "Short-term losses realized (% of initial)": "st_losses",
        "Long-term losses realized (% of initial)": "lt_losses"}
table1 = (T.loc[list(rows.values())] * 100).set_axis(list(rows.keys()))   # in %
table1.loc["Trading cost (bp per year)"] = T.loc["tcost_bp"]
table1.loc["Average number of holdings"] = T.loc["avg_holdings"]
table1.loc["Final NAV ($M)"] = T.loc["final_nav"] / 1e6
table1.loc["Rebalances with gap < 1 bp (%)"] = T.loc["share_gap_below_1bp"] * 100
table1 = table1.astype(float).round(2)
table1.to_csv(FIG + "table1_main.csv")
table1

,S0,S1,S2
Pre-tax return,8.32,8.34,8.27
Index return,8.22,8.22,8.22
"After-tax return, no liquidation",8.41,9.67,9.50
"After-tax return, liquidation",7.60,8.74,8.62
"Tax alpha, no liquidation",0.00,1.25,1.09
"Tax alpha, liquidation",0.00,1.14,1.02
"Tracking error, realized",0.30,0.43,0.38
"Tracking error, ex-ante (mean)",NaN,0.23,0.17
Sales turnover per year,1.07,11.26,11.03
Short-term losses realized (% of initial),0.88,101.98,99.22


## Table 2. Realized losses by period (% of initial capital)

In [3]:
periods = {"2000-04": (2000, 2004), "2005-09": (2005, 2009), "2010-14": (2010, 2014),
           "2015-19": (2015, 2019), "2020-25": (2020, 2025), "Total": (2000, 2025)}
table2 = {}
for k in ["S0", "S1"]:
    a = main[k].annual
    for p, (y0, y1) in periods.items():
        sub = a.loc[y0:y1]
        table2[(k, "short-term")] = table2.get((k, "short-term"), {}) | {p: -sub.st_loss.sum() / 1e4}
        table2[(k, "long-term")] = table2.get((k, "long-term"), {}) | {p: -sub.lt_loss.sum() / 1e4}
table2 = pd.DataFrame(table2).round(1)
table2.to_csv(FIG + "table2_losses_by_period.csv")
table2

S0                   S1          
        short-term long-term short-term long-term
2000-04        0.3       6.0       39.1      20.1
2005-09        0.2      10.2       20.2      10.5
2010-14        0.0       9.1        4.4       0.9
2015-19        0.2       6.4       12.5       2.3
2020-25        0.2       8.5       25.8       5.7
Total          0.9      40.1      102.0      39.5

## Figure 1. After-tax wealth

In [4]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
for k, c in [("S0", "C0"), ("S1", "C1")]:
    rb = main[k].rebal
    axs[0].plot(rb.index, rb.nav / 1e6, color=c, label=f"{k}, no liquidation")
    axs[0].plot(rb.index, after_tax_liquidation_value(main[k]) / 1e6, color=c, ls="--", label=f"{k}, liquidation")
axs[0].set_yscale("log"); axs[0].set_ylabel("$M (log scale)"); axs[0].set_title("After-tax wealth from $1M"); axs[0].legend()

idx = s1.rebal.index
adv_noliq = s1.rebal.nav / s0.rebal.nav.reindex(idx, method="nearest") - 1
adv_liq = after_tax_liquidation_value(s1) / after_tax_liquidation_value(s0).reindex(idx, method="nearest") - 1
axs[1].plot(idx, 100 * adv_noliq, label="no liquidation")
axs[1].plot(idx, 100 * adv_liq, label="liquidation")
axs[1].set_ylabel("%"); axs[1].set_title("S1 after-tax wealth relative to S0"); axs[1].legend()
plt.tight_layout(); save(fig, "fig1_wealth.png"); plt.show()

## Figure 2. Harvested losses

In [5]:
a = s1.annual
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
axs[0].bar(a.index - 0.2, -a.st_loss / 1e4, 0.4, label="short-term")
axs[0].bar(a.index + 0.2, -a.lt_loss / 1e4, 0.4, label="long-term")
axs[0].set_ylabel("% of initial capital"); axs[0].set_title("S1 realized losses per year"); axs[0].legend()
for k in ["S0", "S1"]:
    ann = main[k].annual
    axs[1].plot(ann.index, -(ann.st_loss + ann.lt_loss).cumsum() / 1e4, marker="o", ms=3, label=k)
axs[1].set_ylabel("% of initial capital"); axs[1].set_title("Cumulative realized losses"); axs[1].legend()
plt.tight_layout(); save(fig, "fig2_losses.png"); plt.show()

## Figure 3. Cost basis and unrealized gains

In [6]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
rb = s1.rebal
axs[0].plot(rb.index, rb.mv / 1e6, label="market value")
axs[0].plot(rb.index, rb.basis / 1e6, label="cost basis")
axs[0].set_ylabel("$M"); axs[0].set_title("S1 market value and cost basis"); axs[0].legend()
for k in ["S0", "S1"]:
    r = main[k].rebal
    axs[1].plot(r.index, 100 * r.unrealized / r.nav, label=k)
axs[1].set_ylabel("% of NAV"); axs[1].set_title("Unrealized gain / NAV"); axs[1].legend()
plt.tight_layout(); save(fig, "fig3_basis.png"); plt.show()

## Figure 4. Tracking error, holdings and turnover

In [7]:
r_pre, r_b = pretax_returns(s1), s1.daily.bench_ret
rolling_te = (monthly(r_pre) - monthly(r_b)).rolling(12).std() * np.sqrt(12)
nav_y = s1.daily.nav.groupby(s1.daily.index.year).mean()
sales_y = rb.sells.iloc[1:].groupby(rb.index[1:].year).sum() / nav_y

fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
axs[0].plot(rb.index, 100 * rb.te_ex_ante, label="ex-ante (risk model)")
axs[0].plot(rolling_te.index, 100 * rolling_te, label="realized (12-month rolling)")
axs[0].set_ylabel("%"); axs[0].set_title("S1 tracking error"); axs[0].legend()
axs[1].bar(sales_y.index, 100 * sales_y, color="C1", label="sales (% of NAV)")
axs[1].set_ylabel("% of NAV per year"); axs[1].set_title("S1 turnover and holdings")
ax2 = axs[1].twinx(); ax2.plot(rb.index.year + rb.index.dayofyear / 365, rb.n_hold, color="C0", lw=1, label="stocks held")
ax2.set_ylabel("stocks held"); ax2.grid(False)
axs[1].legend(loc="upper center"); ax2.legend(loc="center right")
plt.tight_layout(); save(fig, "fig4_te_turnover.png"); plt.show()

## Figure 5 and Table 3. Staggered 6-year windows

21 six-year backtests starting each January 2000–2020, each from $1M with everything sold at the window end.
S1 and S2 use gamma_risk = 400 as in the main results; S1 matched uses gamma_risk = 800, whose in-sample
tracking error is closest to S2's.

In [8]:
W = pd.DataFrame([load(os.path.basename(f)[:-4]) for f in glob.glob("results/window_*.pkl")])
P = W.pivot(index="start", columns="strategy")
alpha = lambda conv, k: P[(conv, k)] - P[(conv, "S0")]

fig, ax = plt.subplots(figsize=(10, 3.4))
ax.bar(P.index - 0.2, 100 * alpha("after_tax_liq", "S1"), 0.4, label="liquidation at window end")
ax.bar(P.index + 0.2, 100 * alpha("after_tax_noliq", "S1"), 0.4, label="no liquidation")
ax.set_xlabel("window start (6-year window)"); ax.set_ylabel("% per year"); ax.set_title("S1 tax alpha vs S0"); ax.legend()
plt.tight_layout(); save(fig, "fig5_windows.png"); plt.show()

s1w = pd.DataFrame({"tax alpha, liquidation": alpha("after_tax_liq", "S1"),
                    "tax alpha, no liquidation": alpha("after_tax_noliq", "S1"),
                    "tracking error": P[("te_realized", "S1")]})
(100 * s1w.describe().loc[["mean", "min", "max"]]).round(2)

,"tax alpha, liquidation","tax alpha, no liquidation",tracking error
mean,0.96,1.88,0.40
min,0.22,0.59,0.17
max,3.03,4.42,0.75


In [9]:
# S2 vs S1 in the windows, at about the same tracking error (S1 matched)
rows3 = ["te_realized", "tax_alpha_liq", "tax_alpha_noliq"]
table3 = pd.DataFrame({k: [P[("te_realized", key)].mean(), alpha("after_tax_liq", key).mean(), alpha("after_tax_noliq", key).mean()]
                       for k, key in [("S1", "S1"), ("S1 matched", "S1_matched"), ("S2", "S2")]}, index=rows3)
table3["S2 minus S1 matched"] = table3["S2"] - table3["S1 matched"]
table3 = (100 * table3).round(3)
table3.to_csv(FIG + "table3_windows.csv")
table3

,S1,S1 matched,S2,S2 minus S1 matched
te_realized,0.398,0.313,0.277,-0.036
tax_alpha_liq,0.959,0.835,0.747,-0.089
tax_alpha_noliq,1.884,1.685,1.385,-0.300


## Figure 6 and Table 4. Tracking-error frontier, 2000–2025

Separate experiment: S1 and S2 with a cap on the ex-ante tracking error and gamma_risk = 10, so that the cap sets the risk level.

In [10]:
CAPS = [0.0025, 0.005, 0.0075, 0.01, 0.015, 0.02]
s0_sum = summary(s0)
def run_summary(key, cap):
    return load(f"te_full_{key}_{cap:.4f}")
rows = []
for cap in CAPS:
    for key in ["S1", "S2"]:
        s = run_summary(key, cap)
        rows.append(dict(cap=cap, strategy=key, te_ex_ante=s["te_ex_ante"], te_realized=s["te_realized"],
                         tax_alpha_liq=s["after_tax_liq"] - s0_sum["after_tax_liq"],
                         tax_alpha_noliq=s["after_tax_noliq"] - s0_sum["after_tax_noliq"],
                         after_tax_liq=s["after_tax_liq"], sell_turnover=s["sell_turnover"], tcost_bp=s["tcost_bp"]))
F = pd.DataFrame(rows)

fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
for key, c in [("S1", "C0"), ("S2", "C1")]:
    f = F[F.strategy == key]
    axs[0].plot(100 * f.cap, 100 * f.tax_alpha_liq, "o-", color=c, label=f"{key}, liquidation")
    axs[0].plot(100 * f.cap, 100 * f.tax_alpha_noliq, "o--", color=c, label=f"{key}, no liquidation")
    axs[1].plot(100 * f.cap, 100 * f.te_realized, "o-", color=c, label=f"{key}, realized")
    axs[1].plot(100 * f.cap, 100 * f.te_ex_ante, "o--", color=c, label=f"{key}, ex-ante (mean)")
axs[0].set_xlabel("tracking-error cap (%)"); axs[0].set_ylabel("tax alpha vs S0 (% per year)"); axs[0].set_title("Tax alpha"); axs[0].legend()
axs[1].plot(100 * F.cap.unique(), 100 * F.cap.unique(), color="C2", lw=1, ls=":", label="cap")
axs[1].set_xlabel("tracking-error cap (%)"); axs[1].set_ylabel("%"); axs[1].set_title("Tracking error"); axs[1].legend()
plt.tight_layout(); save(fig, "fig6_frontier.png"); plt.show()

table4 = F.set_index(["cap", "strategy"])
table4[["te_ex_ante", "te_realized", "tax_alpha_liq", "tax_alpha_noliq", "after_tax_liq", "sell_turnover"]] *= 100
table4.index = table4.index.set_levels([[f"{100 * c:g}%" for c in table4.index.levels[0]], table4.index.levels[1]])
table4 = table4.round(2)
table4.to_csv(FIG + "table4_frontier.csv")
table4

te_ex_ante  te_realized  tax_alpha_liq  tax_alpha_noliq  \
cap   strategy                                                            
0.25% S1              0.25         0.40           1.15             1.26   
      S2              0.25         0.41           1.17             1.26   
0.5%  S1              0.40         0.60           1.33             1.46   
      S2              0.39         0.59           1.35             1.45   
0.75% S1              0.51         0.71           1.42             1.56   
      S2              0.49         0.71           1.39             1.49   
1%    S1              0.66         0.93           1.60             1.74   
      S2              0.58         0.88           1.49             1.58   
1.5%  S1              0.87         1.17           1.70             1.85   
      S2              0.81         1.17           1.78             1.85   
2%    S1              1.04         1.36           1.87             2.01   
      S2              0.88         1.24           1.71             1.79   

                after_tax_liq  sell_turnover  tcost_bp  
cap   strategy                                          
0.25% S1                 8.75          12.47      1.40  
      S2                 8.77          12.55      1.41  
0.5%  S1                 8.93          13.48      1.51  
      S2                 8.96          14.19      1.58  
0.75% S1                 9.02          14.71      1.64  
      S2                 9.00          14.87      1.65  
1%    S1                 9.21          14.70      1.63  
      S2                 9.10          15.43      1.71  
1.5%  S1                 9.30          15.65      1.73  
      S2                 9.39          18.26      1.99  
2%    S1                 9.47          15.52      1.73  
      S2                 9.31          17.70      1.94

## Optimality gap (numbers quoted in the text)

In [11]:
g = s1.rebal.gap_bp.iloc[1:]
print(f"S1: {len(g)} rebalances, median gap {g.median():.4f} bp, max {g.max():.2f} bp, below 1 bp in {(g < 1).mean():.1%}")
print(f"S1 losses realized 2000-2004: {-(s1.annual.loc[2000:2004, ['st_loss', 'lt_loss']].sum().sum()) / 1e6:.0%} of initial capital")

S1: 301 rebalances, median gap 0.0001 bp, max 1.37 bp, below 1 bp in 99.7%
S1 losses realized 2000-2004: 59% of initial capital
